# Hosting physrisk

The physrisk library is designed to be hosted as a service. The [physrisk-api project](https://github.com/os-climate/physrisk-api) provides an example of how this can be done.

## Performance and concurrency design

An important function of physrisk is to obtain asset-level hazard indicators efficiently from compressed, chunked stores (Zarr, Cloud Optimized GeoTIFF etc.) and from external APIs. [Zarr](https://zarr.readthedocs.io/en/stable/) is the main chunked store format.

To obtain chunked data efficiently, physrisk:

1. identifies all hazard indicator data needed by the calculation, so that it can be retrieved in one go;
2. converts asset locations to array indices (using each array's affine transform);
3. has Zarr identify, download and decode the chunks containing those indices, retaining only the values needed;
4. applies any interpolation to those values.

### Zarr reads

All Zarr reads in the process, for hazard models and map tiles alike, run on a single, dedicated event loop, shared by every calculation and tile request in flight (for example, concurrent requests handled by FastAPI). Chunks are downloaded asynchronously on this loop and decoded on its thread pool. Because the loop is shared:

- the decode thread pool, sized by default to the CPUs available to the process (respecting container CPU quotas and CPU affinity), bounds the number of chunks being decoded at once, and with it CPU use and memory for decoded chunks;
- the number of chunks in flight, i.e. being downloaded, awaiting decode or being decoded, is limited across the process. Each chunk holds one of a fixed number of slots from download until it is decoded, which bounds the memory used by downloaded and decoded chunks however many arrays, calculations and tiles are in flight. physrisk does this by configuring Zarr to use a codec pipeline that adds the limit to Zarr's default pipeline;
- the download connection pool and the cache of opened arrays are shared across calculations.

Zarr's own `async.concurrency` setting limits the chunks in flight for each array read separately. physrisk sets it to the same value as the process-wide limit, so that a single read can use all of the slots and the process-wide limit is the one that applies.

Slots are granted in the order chunks are requested, so a tile or a small calculation that starts while a large calculation is in progress waits behind the chunks the large calculation has already requested.

These can be set via the container configuration:

| Setting | Default | Effect |
|---|---|---|
| `zarr_decode_threads` | CPUs available | Threads decoding chunks |
| `zarr_max_chunks_in_flight` | max(16, 8 × CPUs) | Chunks in flight across the process; reduce if memory is tight |
| `zarr_cache_arrays` | `True` | Keep opened arrays so metadata is read once per array |

### Hazard models

Each calculation runs its hazard models on its own event loop. `get_hazard_data_async` is the primary method; `get_hazard_data` runs it to completion for synchronous callers. Composite models gather their component models concurrently: natively asynchronous models (such as the Zarr and JBA models) run on the calculation's loop, while synchronous-only models run on worker threads so as not to block it. Zarr reads are handed to the shared Zarr loop described above.

### Map tiles

Tiles created from Zarr data read through the same Zarr reader, and so share the Zarr loop and its limits; rendering the image runs on the calling thread. Tiles from JBA run on a separate process-level loop so that a single HTTP session, and its connections, is reused across tiles.

### Limiting concurrent requests

The limits above bound memory used by reads in progress, but not each calculation's own working memory (assets, hazard data, impact results), which grows with portfolio size. Limiting how many calculations run at once, and keeping tiles responsive while they run (for example, by serving tiles from separate worker processes), is therefore the responsibility of the hosting layer: physrisk-api, for example, can run asset impact requests through a dedicated limiter. Note that all limits are per process: with several worker processes, each has its own Zarr loop and decode pool.